# Calibrate a mass from force and speed measurements

Write the data, physical model, solver and parameter update in this notebook. No example module runs the experiment for you.

Use a Python kernel with this **development checkout** installed (`python -m pip install .` from its repository root). The released 0.1.3 wheel predates these APIs. This notebook uses CPU execution and the Python standard library; it does not require plotting or dataframe packages. Hosted Colab execution is not yet verified.

We assume a body starts at rest and obeys $m\,dv/dt=F(t)$. Force is measured in millinewtons, time in milliseconds and speed in millimetres per second. Only mass is unknown; the force calibration and time origin are known. These synthetic, noise-free measurements have an independently known answer, so we can check the entire calibration path.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import eqiora as q
from eqiora import data, units as u

work = TemporaryDirectory(prefix="eqiora-measured-force-")
root = Path(work.name)
print("Eqiora", q.__version__)
print("Working files:", root)

## 1. Map the columns and check their dimensions

CSV parsing does not infer units, align clocks, sort rows or fill missing values. Each column below has an explicit input unit. The native converter combines its exact decimal value and unit scale before rounding to binary64.

The force rises linearly from 0 N to 2 N at 0.5 s, then falls to zero at 1 s. The speed samples belong to a 2 kg body. Keep the original sources and their column mapping with the package.

In [ ]:
force_csv = "elapsed_ms,force_mN\n0,0\n500,2000\n1000,0\n"
speed_csv = "elapsed_ms,speed_mm_s\n0,0\n125,15.625\n500,250\n875,484.375\n1000,500\n"
force_samples, force_dimensions = data.csv_array(force_csv, columns=(
    ("elapsed_ms", u.s.prefixed("m")), ("force_mN", u.N.prefixed("m")),
))
speed_samples, speed_dimensions = data.csv_array(speed_csv, columns=(
    ("elapsed_ms", u.s.prefixed("m")), ("speed_mm_s", u.m.prefixed("m") / u.s),
))
assert force_dimensions == (q.Dimension(time=1), q.Dimension(mass=1, length=1, time=-2))
assert speed_dimensions == (q.Dimension(time=1), q.Dimension(length=1, time=-1))
output_times = tuple(speed_samples.values[::2])[1:]
print("Force array:", force_samples.shape, force_samples.values)
print("Speed sample times (s):", output_times)

## 2. Write the model and the comparison once

Both histories use the existing property-table machinery. We explicitly choose piecewise-affine interpolation, reject missing data and extrapolation, and reject time derivatives at knots and endpoints. Duplicate or unordered time coordinates fail table admission.

The fitted `mass` belongs to the Model. The Component receives that same Parameter, so changing it does not require rewriting source. Define the speed residual in m/s, divide by an explicit uncertainty of 0.01 m/s, then form a dimensionless least-squares loss. Observable composition and its derivatives use these definitions throughout.

In [ ]:
source = r"""
public property contract ForceHistory(input elapsed:s): N {
    derivatives first_open_intervals;
}
public property release Samples: ForceHistory {
    table {
        data force_samples;
        axis elapsed:s;
        value force:N;
        interpolation piecewise_affine;
        preprocessing identity;
        missing reject;
        knot_derivative reject;
        endpoint_derivative reject;
    }
    validity elapsed in [0[s],1[s]];
    outside reject;
    branch single;
    citation synthetic_definition;
    license repository_license;
}
public property contract SpeedHistory(input elapsed:s): m/s {
    derivatives first_open_intervals;
}
public property release SpeedSamples: SpeedHistory {
    table {
        data speed_samples;
        axis elapsed:s;
        value speed:m/s;
        interpolation piecewise_affine;
        preprocessing identity;
        missing reject;
        knot_derivative reject;
        endpoint_derivative reject;
    }
    validity elapsed in [0[s],1[s]];
    outside reject;
    branch single;
    citation synthetic_definition;
    license repository_license;
}
component Driven(parameter mass:kg, property force:ForceHistory, property measured:SpeedHistory) {
    state velocity:m/s;
    initial { velocity=0[m/s]; }
    relation motion { derivative(velocity)=force(elapsed=time())/mass; }
    observable speed:m/s=velocity;
    observable residual:m/s=velocity-measured(elapsed=time());
    observable normalized:1=residual / 0.01[m/s];
    observable loss:1=normalized*normalized/2;
}
public model Main() {
    parameter mass:kg=2;
    instance body:Driven(mass=mass, force=Samples, measured=SpeedSamples);
}
"""

## 3. Resolve the exact package

The existing package store binds the model source, normalized arrays, policies and attribution. An array digest identifies its normalized values and shape; the raw CSV and mapping are retained in the cited documentation.

In [ ]:
project = root / "project"
for directory in ("src", "data", "docs"):
    (project / directory).mkdir(parents=True)
(project / "eqiora.toml").write_text(
    '[package]\nname="org.example.MeasuredForce"\nversion="1.0.0"\nentry="main"\n'
)
(project / "src/main.eqi").write_text(source)
(project / "data/force_samples.json").write_bytes(force_samples.to_bytes())
(project / "data/speed_samples.json").write_bytes(speed_samples.to_bytes())
(project / "docs/synthetic_definition.md").write_text(
    "Synthetic force (ms,mN) and speed (ms,mm/s); columns retained in that order:\n\n"
    + force_csv + "\n" + speed_csv
)
(project / "docs/repository_license.md").write_text("CC0-1.0 synthetic data.")
store = root / "store"
store.mkdir()
resolution = q.resolve_local_project(project, store)
model = q.compile_package(store, resolution, entry="Main")
mass_id = model.parameter("mass").id
velocity_id = model.field("body.velocity").id
observable_ids = {name: model.observable("body." + name).id
                  for name in ("speed", "residual", "normalized", "loss")}
assert model.parameter("body.mass").id == mass_id
original_model = model.to_bytes()
model = q.Model.from_bytes(original_model)
print("Resolved force identity:", force_samples.digest)

## 4. Solve at the recorded times and retain Parameter products

Each candidate is an immutable Model edit followed by an ordinary Plan and Run. The solver retains $\partial v/\partial m$ at the requested output times. Tolerances for that derivative have units $(\mathrm{m/s})/\mathrm{kg}$.

We vary mass while holding the recorded times fixed. A force-table knot is therefore allowed here: we do not ask for a derivative with respect to the table's time coordinate.

In [ ]:
def solve_mass(mass_kg):
    if mass_kg <= 0:
        raise ValueError("This physical experiment requires a positive mass")
    initial_mass = model.parameter(mass_id)
    candidate = model if mass_kg == initial_mass.value else model.commit(
        model.preview_value_edit(mass_id, mass_kg)
    )
    mass = candidate.parameter(mass_id)
    velocity = candidate.field(velocity_id)
    forward = q.time.ForwardSensitivity(relative_tolerance=1e-10, absolute_tolerances=(
        q.time.SensitivityTolerance((velocity, 0, 0, False), mass, 1e-12,
                                    q.Dimension(mass=-1, length=1, time=-1)),
    ))
    plan = q.resolve(candidate, temporal=q.time.Tsitouras45(
        initial_step_s=1e-3, relative_tolerance=1e-11,
        absolute_tolerances={(velocity, 0, 0, False): 1e-13},
        forward_sensitivities=forward,
    ))
    result = q.run(plan, state=q.State.initial(plan), until_s=1., output_times_s=output_times)
    return candidate, plan, result


def compare(candidate, result):
    mass = candidate.parameter(mass_id)
    direction = {mass: (q.Dimension(mass=1), 1.)}
    normalized = candidate.observable(observable_ids["normalized"])
    loss = candidate.observable(observable_ids["loss"])
    residuals, products, losses = [], [], []
    for index in range(len(output_times)):
        residual = result.observe(normalized, output_index=index)
        product = result.observe_parameter_jvp(normalized, direction, output_index=index)
        term = result.observe(loss, output_index=index)
        assert residual.value_type == product.value_type == term.value_type == q.ValueType.real(q.Dimension())
        residuals.append(residual.value)
        products.append(product.value)
        losses.append(term.value)
    return residuals, products, sum(losses)

## 5. Fit the mass with a scalar Gauss–Newton update

Let $r_i$ be the normalized residual and $J_i$ its response to a 1 kg mass direction. Linearizing the residual gives the least-squares update, in kg,

$$\Delta m=-\frac{\sum_i J_i r_i}{\sum_i J_i^2}.$$

This is an explicit update for this positive one-parameter example. It is not a general constrained optimizer. We reject an unidentifiable zero denominator or nonpositive trial mass. The data and weights are fixed; their derivatives are not added to the model response.

In [ ]:
mass_kg = 1.
history = []
for iteration in range(8):
    candidate, plan, result = solve_mass(mass_kg)
    residuals, products, loss = compare(candidate, result)
    history.append((iteration, mass_kg, loss))
    print(f"{iteration}: mass={mass_kg:.10f} kg, loss={loss:.6g}")
    if loss < 1e-12:
        break
    denominator = sum(value * value for value in products)
    if denominator <= 0:
        raise ValueError("These observations do not identify the mass")
    step_kg = -sum(r * j for r, j in zip(residuals, products)) / denominator
    mass_kg += step_kg
else:
    raise RuntimeError("The declared eight-iteration budget did not converge")

fitted_mass = mass_kg
assert abs(fitted_mass - 2.) < 1e-7
assert model.to_bytes() == original_model

## 6. Check an independent answer and reopen the accepted result

Integrating the triangular force by hand gives the impulse

$$I(t)=\begin{cases}2t^2,&t\leq1/2,\\4t-2t^2-1,&t>1/2.\end{cases}$$

Thus $v=I/m$ and the response to a 1 kg direction is $-I/m^2$. Check both against the numerical result. Reopening retains the exact Model/Plan/Result identities and policies; source aliases are authoring metadata, so select the saved IDs after reopening.

In [ ]:
speed = candidate.observable(observable_ids["speed"])
mass = candidate.parameter(mass_id)
for index, time in enumerate(output_times):
    impulse = 2*time*time if time <= .5 else 4*time-2*time*time-1
    value = result.observe(speed, output_index=index)
    product = result.observe_parameter_jvp(speed, {mass: (q.Dimension(mass=1), 1.)}, output_index=index)
    assert abs(value.value - impulse/fitted_mass) < 1e-8
    assert abs(product.value + impulse/fitted_mass**2) < 1e-8
    print(f"t={time:g} s: speed={value.value:.8f} m/s, response={product.value:.8f} m/s")

for name, artifact in (("model", candidate), ("plan", plan), ("result", result)):
    (root / (name + ".json")).write_bytes(artifact.to_bytes())
reopened_model = q.Model.from_bytes((root / "model.json").read_bytes())
reopened_plan = q.Plan.from_bytes((root / "plan.json").read_bytes())
reopened = q.Result.from_bytes(reopened_plan, (root / "result.json").read_bytes())
assert reopened.to_bytes() == result.to_bytes()
assert compare(reopened_model, reopened) == compare(candidate, result)

## Try changing the experiment

- Add different recorded times and derive their speed values from the impulse formula. Update both the table and output schedule; do not substitute arbitrary-time interpolation for a retained output.
- Duplicate a force timestamp or delete a selected cell. Observe rejection rather than silent sorting, averaging or filling.
- Change a column's unit to an incompatible dimension. The explicit dimension check should stop the workflow before the data is bound to a property.
- Change the declared speed uncertainty. The dimensionless loss and sensitivities rescale together; the best mass for these noise-free data is unchanged.
- Add noise and ask whether the residuals agree with the assumed uncertainty. A low loss cannot by itself prove that the physical model or instrument calibration is correct.

This lumped example does not claim spatial sensor support, general dataframe conversion, differentiation of a knot time, uncertainty estimation or a reusable optimizer.